# Stage A：Tucker・コア加算・xシフトの数値整合性

このノートブックは `stage_a_validation.py` の入口です。2048×2048、下のセルで指定するrank、50 µm、全Frozen-Waveモードを用います。数値窓は物理開口ではありません。結果は `output_dir` に保存します。実機SLM、4f、Gaussian照明、50 cm実伝搬は評価しません。

オンラインでは全画素のIFFTを行わず、事前にIFFTした因子行列 Vy, Vx と加算済みコアから E = Vy @ C @ Vx.T で直接復元します。従来経路との一致誤差と、ウォームアップ後3回の復元時間の中央値も保存します。


In [ ]:
import importlib
import stage_a_validation as stage_a
stage_a = importlib.reload(stage_a)
config = stage_a.StageAConfig(rank=100)
modes = stage_a.design_modes(config)
output_dir = 'stage_a_outputs_rank100_spatial_v4'  # 旧結果は残し、新指標で再計算する
print('grid:', config.grid_size)
print('rank:', config.rank)
print('mode count:', len(modes.m))
print('m:', modes.m)


In [ ]:
from dataclasses import asdict
from pathlib import Path
import json
import importlib
import stage_a_validation as stage_a
stage_a = importlib.reload(stage_a)  # Notebookカーネルに残った旧コードを更新
print('検証コード:', stage_a.__file__)
if stage_a.METRIC_SCHEMA_VERSION < 4:
    raise RuntimeError('古い検証コードが読み込まれています。Notebookのカーネルを再起動してください。')

result_dir = Path(output_dir)
summary_file = result_dir / 'summary.json'
manifest_file = result_dir / 'manifest.json'
if summary_file.is_file() and manifest_file.is_file():
    saved = json.loads(summary_file.read_text(encoding='utf-8'))
    if (saved.get('stage') == 'A_algorithmic_validation_only'
            and saved.get('config') == asdict(config)
            and saved.get('metric_schema_version', 1) >= stage_a.METRIC_SCHEMA_VERSION
            and 'complex_snr_db' in saved.get('coaxial', {}).get('spatial', {})):
        summary = saved
        print('同じ条件の完了結果を読み込みました。再計算はしていません。')
    else:
        raise ValueError('保存済み結果は旧評価指標または別条件です。1番目のセルで output_dir を新しい名前に変更して再計算してください。')
elif result_dir.exists() and any(result_dir.iterdir()):
    raise ValueError('途中結果が残っています。1番目のセルで output_dir を新しい名前に変更してください。')
else:
    summary = stage_a.run_stage_a(output_dir, config)
    print('新しい計算を実行しました。')
if (summary.get('metric_schema_version', 0) < stage_a.METRIC_SCHEMA_VERSION
        or 'complex_snr_db' not in summary.get('coaxial', {}).get('spatial', {})):
    raise RuntimeError('計算結果に新しいSNRがありません。検証コードの読み込み先を確認してください。')
print('結果:', result_dir.resolve())
print('同軸の複素場相対L2誤差:', summary['coaxial']['spatial']['complex_relative_l2'])
print('同軸の複素場SNR[dB]:', summary['coaxial']['spatial']['complex_snr_db'])
print('同軸の振幅SSIM / PSNR[dB]:', summary['coaxial']['spatial']['amplitude_ssim'], summary['coaxial']['spatial']['amplitude_psnr_db'])
print('同軸の強度SSIM / PSNR[dB]:', summary['coaxial']['spatial']['intensity_ssim'], summary['coaxial']['spatial']['intensity_psnr_db'])
print('同軸CGHの8-bit SSIM[参考] / 円周PSNR[dB]:', summary['coaxial']['cgh']['raw_8bit_code_ssim_wrap_sensitive'], summary['coaxial']['cgh']['circular_code_psnr_db'])
print('xシフト後の複素場相対L2誤差:', summary['shifted']['spatial']['complex_relative_l2'])
print('xシフト後の複素場SNR[dB]:', summary['shifted']['spatial']['complex_snr_db'])
print('xシフト後の振幅SSIM / PSNR[dB]:', summary['shifted']['spatial']['amplitude_ssim'], summary['shifted']['spatial']['amplitude_psnr_db'])
print('xシフト後の強度SSIM / PSNR[dB]:', summary['shifted']['spatial']['intensity_ssim'], summary['shifted']['spatial']['intensity_psnr_db'])
print('xシフト後CGHの8-bit SSIM[参考] / 円周PSNR[dB]:', summary['shifted']['cgh']['raw_8bit_code_ssim_wrap_sensitive'], summary['shifted']['cgh']['circular_code_psnr_db'])
print('シフト直接法のBessel場再計算数:', summary['shifted_direct_beam_evaluations'])
print('シフト直接法の場生成時間[s]:', summary['timings_seconds']['shifted_online_seconds']['direct_bessel_recomputation_and_sum_seconds'])
print('シフト提案法の場生成時間[s]:', summary['timings_seconds']['shifted_online_seconds']['proposed_field_generation_seconds'])
print('実空間シフトと周期FFTシフトの複素場相対L2差:', summary['shifted']['periodic_shift_model']['direct_vs_periodic_spatial_complex_relative_l2'])
for case in ('coaxial', 'shifted'):
    benchmark = summary['timings_seconds'][case + '_reconstruction_benchmark']
    print(case, '従来のFFT復元+IFFTの中央値[s]:', benchmark['legacy_median_seconds'])
    print(case, '実空間因子による復元の中央値[s]:', benchmark['spatial_factor_median_seconds'])
    print(case, '復元部分の高速化倍率:', benchmark['reconstruction_speedup'])
    print(case, '2経路の複素場相対L2差:', benchmark['spatial_factor_vs_fft_ifft_complex_relative_l2'])
